In [ ]:
!pip install langchain chromadb pypdf langchain-google-genai langchain-groq langgraph langchain-community langchain-text-splitters google-cloud-aiplatform langchain-google-vertexai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata
gemini_api_key=userdata.get('GEMINI_KEY')

# 1. Set your API key for Google Gemini (used for embeddings)
os.environ["GOOGLE_API_KEY"] = gemini_api_key

In [ ]:
pdf_files = [


    "ARP.pdf",
    "CRC.pdf",
    "ETHERNET.pdf",
    "ev.pdf",


]
PERSIST_DIR = "./chroma_db"

In [ ]:
# 2. Safe File Loading 32,135 words 195,154 characters
all_docs = []

for file_path in pdf_files:
    print(f"Attempting to load: {file_path}...")
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        if not docs:
            print(f"Warning: {file_path} was loaded, but no text was found.")
        else:
            all_docs.extend(docs)
            print(f"Successfully loaded {len(docs)} pages from this part.")
    except Exception as e:
        print(f"Error loading the PDF: {e}")

print(f"\nDone! Combined total of {len(all_docs)} pages loaded into memory.")

Attempting to load: ARP.pdf...
Successfully loaded 14 pages from this part.
Attempting to load: CRC.pdf...
Successfully loaded 18 pages from this part.
Attempting to load: ETHERNET.pdf...
Successfully loaded 20 pages from this part.
Attempting to load: ev.pdf...
Successfully loaded 52 pages from this part.

Done! Combined total of 104 pages loaded into memory.


In [ ]:
import re

# ==========================================
# Sentence Chunk Generator
# ==========================================

try:
    print("Generating sentence chunks from documents...")

    CHUNK_SIZE = 5
    CHUNK_OVERLAP = 2

    if CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError(
            "CHUNK_OVERLAP should be less than CHUNK_SIZE"
        )

    generated_chunks = []

    jump = CHUNK_SIZE - CHUNK_OVERLAP

    for page_doc in all_docs:

        try:
            raw_text = page_doc.page_content

            # Basic text normalization
            cleaned_text = raw_text.replace("\n", " ")
            cleaned_text = cleaned_text.replace("\r", " ")
            cleaned_text = cleaned_text.replace("\t", " ")

            cleaned_text = re.sub(
                r"\s+",
                " ",
                cleaned_text
            )

            cleaned_text = re.sub(
                r"\s+([.,!?;:])",
                r"\1",
                cleaned_text
            ).strip()

            if cleaned_text == "":
                print(
                    f"Skipped empty content in "
                    f"{page_doc.metadata.get('source', 'Unknown File')} "
                    f"(Page {page_doc.metadata.get('page')})"
                )
                continue

            # Sentence extraction
            sentence_list = re.split(
                r"(?<=[.!?])\s+",
                cleaned_text
            )

            sentence_list = list(
                filter(
                    None,
                    [s.strip() for s in sentence_list]
                )
            )

            if not sentence_list:

                print(
                    f"No sentences detected in "
                    f"{page_doc.metadata.get('source', 'Unknown File')} "
                    f"(Page {page_doc.metadata.get('page')})"
                )

                generated_chunks.append(
                    page_doc.__class__(
                        page_content=cleaned_text,
                        metadata=page_doc.metadata
                    )
                )

                continue

            position = 0

            while position < len(sentence_list):

                section = sentence_list[
                    position:position + CHUNK_SIZE
                ]

                if section:

                    generated_chunks.append(
                        page_doc.__class__(
                            page_content=" ".join(section),
                            metadata=page_doc.metadata
                        )
                    )

                position += jump

        except Exception as err:

            print(
                f"Failed on "
                f"{page_doc.metadata.get('source', 'Unknown File')} "
                f"(Page {page_doc.metadata.get('page')}): {err}"
            )

            continue

    chunks = generated_chunks

    print("\nChunk generation finished.")
    print(f"Chunks produced: {len(chunks)}")

except ValueError as err:
    print(f"Invalid configuration: {err}")

except Exception as err:
    print(f"Chunking process terminated: {err}")
    exit()

Generating sentence chunks from documents...
Skipped empty content in CRC.pdf (Page 10)
Skipped empty content in CRC.pdf (Page 13)
Skipped empty content in CRC.pdf (Page 17)
Skipped empty content in ETHERNET.pdf (Page 7)
Skipped empty content in ETHERNET.pdf (Page 8)
Skipped empty content in ETHERNET.pdf (Page 15)

Chunk generation finished.
Chunks produced: 191


In [ ]:
# update the chroma database
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Old ChromaDB deleted.")

In [ ]:
import time
from langchain_community.vectorstores import Chroma

# ==========================================
# Vector Database Population
# ==========================================

try:
    print("Setting up embedding model and vector database...")

    embedding_model = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    db = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embedding_model
    )

    MAX_BATCH = 50
    document_count = len(chunks)

    print(
        f"Processing {document_count} chunks "
        f"for vector storage..."
    )

    batch_start = 0

    while batch_start < document_count:

        batch_end = min(
            batch_start + MAX_BATCH,
            document_count
        )

        current_batch = chunks[batch_start:batch_end]

        print(
            f"Adding records "
            f"{batch_start} - {batch_end}"
        )

        db.add_documents(current_batch)

        batch_start = batch_end

        if batch_start < document_count:
            print(
                "Rate limit protection enabled. "
                "Sleeping for 60 seconds..."
            )
            time.sleep(60)

    print(
        "Embedding completed successfully. "
        "Data has been saved to disk."
    )

except Exception as error:

    print(
        f"Vector store creation failed: {error}"
    )

Setting up embedding model and vector database...


/tmp/ipykernel_9629/835759169.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


Processing 191 chunks for vector storage...
Adding records 0 - 50
Rate limit protection enabled. Sleeping for 60 seconds...
Adding records 50 - 100
Rate limit protection enabled. Sleeping for 60 seconds...
Adding records 100 - 150
Rate limit protection enabled. Sleeping for 60 seconds...
Adding records 150 - 191
Embedding completed successfully. Data has been saved to disk.


In [ ]:
# Zip the chroma_db folder so you can download it
!zip -r phase2.zip ./chroma_db

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/ (stored 0%)
  adding: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/header.bin (deflated 63%)
  adding: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/link_lists.bin (stored 0%)
  adding: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/length.bin (deflated 70%)
  adding: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/data_level0.bin (deflated 100%)
  adding: chroma_db/chroma.sqlite3 (deflated 26%)


In [ ]:
from google.colab import files
files.download("phase2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Unzip the database back into the environment
!unzip phase2.zip

Archive:  phase2.zip
replace chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/header.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/header.bin  
replace chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/link_lists.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
 extracting: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/link_lists.bin  
replace chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/length.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/length.bin  
replace chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/data_level0.bin? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chroma_db/28d8e96f-017d-48bf-a25e-96cf9c986f8f/data_level0.bin  
replace chroma_db/chroma.sqlite3? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: chroma_db/chroma.sqlite3  


In [ ]:
import os
from typing import List
from typing_extensions import TypedDict

from google.colab import userdata

from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

from langgraph.graph import StateGraph, START, END

# ==========================================
# Configuration
# ==========================================

GOOGLE_TOKEN = userdata.get("GEMINI_KEY")
GROQ_TOKEN = userdata.get("GROQ_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_TOKEN
os.environ["GROQ_API_KEY"] = GROQ_TOKEN

DB_PATH = "./chroma_db"

# ==========================================
# Load Models & Database
# ==========================================

try:
    embedding_engine = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    chroma_db = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embedding_engine
    )

    search_engine = chroma_db.as_retriever(
        search_kwargs={"k": 2}
    )

    chat_model = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )

    print("Resources loaded successfully.")

except Exception as error:

    print(f"Startup failed: {error}")
    exit()

# ==========================================
# State Definition
# ==========================================

class PipelineState(TypedDict):
    question: str
    context: List[str]
    generation: str

# ==========================================
# Retrieval Stage
# ==========================================

def fetch_context(state: PipelineState):

    print("Retrieval step running...")

    try:
        query = state["question"]

        retrieved_docs = search_engine.invoke(query)

        extracted_text = [
            item.page_content
            for item in retrieved_docs
        ]

        return {
            "context": extracted_text
        }

    except Exception as error:

        print(f"Retrieval failure: {error}")

        return {
            "context": [
                "Error: Unable to access document context."
            ]
        }

# ==========================================
# Generation Stage
# ==========================================

def build_answer(state: PipelineState):

    print("Generation step running...")

    try:
        user_query = state["question"]
        retrieved_context = state["context"]

        context_block = "\n\n---\n\n".join(
            retrieved_context
        )

        prompt = PromptTemplate(
            input_variables=[
                "context",
                "question"
            ],
            template="""
You are a helpful academic assistant.

Answer the user's question strictly using the supplied context.

If the answer cannot be determined from the provided information,
clearly state that the answer is not available in the documents.

Context:
{context}

Question:
{question}

Answer:
"""
        )

        chain = prompt | chat_model

        result = chain.invoke(
            {
                "context": context_block,
                "question": user_query
            }
        )

        return {
            "generation": result.content
        }

    except Exception as error:

        print(f"Generation failure: {error}")

        return {
            "generation":
            "Error: Failed while generating a response."
        }

# ==========================================
# Workflow Construction
# ==========================================

graph_builder = StateGraph(PipelineState)

graph_builder.add_node(
    "retrieve",
    fetch_context
)

graph_builder.add_node(
    "generate",
    build_answer
)

graph_builder.add_edge(
    START,
    "retrieve"
)

graph_builder.add_edge(
    "retrieve",
    "generate"
)

graph_builder.add_edge(
    "generate",
    END
)

app = graph_builder.compile()

print("Workflow compilation completed.")

Resources loaded successfully.
Workflow compilation completed.


In [ ]:
# Define your test question
input_data = {"question": "What is ARP Operation"}

# Run the pipeline
try:
    final_state = app.invoke(input_data)

    print("\n" + "="*40)
    print("FINAL RAG RESPONSE:")
    print("="*40)
    print(final_state["generation"])

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

Retrieval step running...
Generation step running...

FINAL RAG RESPONSE:
ARP Operation is a process that involves the following steps: 
1. The sender knows the IP address of the target. 
2. IP asks ARP to create an ARP request message, filling in the sender physical address, the sender IP address, and the target IP address, with the target physical address field filled with Os. 

Note: The context only provides two steps of the ARP operation, as step 3 is not included in the given information.


In [ ]:
!pip install ragas datasets pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 22.7 MB/s eta 0:00:00


In [ ]:
import sys
import types
import langchain_community.llms

# 1. Create a dummy module to replace the missing VertexAI path
mock_vertex_module = types.ModuleType("langchain_community.chat_models.vertexai")

# 2. Add a hollow ChatVertexAI class to it
mock_vertex_module.ChatVertexAI = type("ChatVertexAI", (object,), {})

# 3. Inject our dummy module directly into Python's system modules
sys.modules["langchain_community.chat_models.vertexai"] = mock_vertex_module

# 4. Patch the legacy llms module as well to prevent the next error on line 13
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

print("Hotfix applied to system memory. You can now import RAGAS!")

Hotfix applied to system memory. You can now import RAGAS!


In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate

# 1. NEW IMPORTS: Import the capitalized Class names directly
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# 2. Import the Ragas wrappers for Langchain objects
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# --- 1. Define your test questions and their expected answers ---
test_questions = [
    "What is the difference between error-correcting codes and error-detecting codes?",
    "What are the three linear systematic block codes discussed for error detection?",
    "What is a Cyclic Redundancy Check (CRC)?",
    "How is a bit string represented in CRC polynomial coding?",
    "What is the role of the generator polynomial G(x) in CRC?",
    "What are the steps involved in CRC checksum generation?",
    "What is the purpose of the Medium Access Control (MAC) sublayer?",
    "What is the difference between static and dynamic channel allocation?",
    "What are the three major categories of multiple access protocols?",
    "How does Pure ALOHA handle collisions?",
    "How does Slotted ALOHA improve upon Pure ALOHA?",
    "What is the basic principle of Carrier Sense Multiple Access (CSMA)?",
    "How does Non-Persistent CSMA reduce collisions compared to 1-Persistent CSMA?",
    "What is the purpose of ARP?",
    "What is the difference between logical and physical addresses?",
    "How does ARP resolve a physical address from an IP address?",
    "What information is contained in an ARP request message?",
    "What is ARP caching and why is it useful?",
    "What is the purpose of the Ethernet preamble and Start of Frame Delimiter?",
    "How does the Binary Exponential Backoff algorithm work in Ethernet?"
]





ground_truths = [
     "Error-correcting codes include enough redundant information for the receiver to determine the original transmitted data, while error-detecting codes include enough redundancy only to detect errors and require retransmission.",

     "The three linear systematic block codes discussed are Parity, Checksums, and Cyclic Redundancy Checks (CRCs).",

     "A Cyclic Redundancy Check (CRC) is an error-detection technique that treats bit strings as polynomials and uses modulo-2 arithmetic to detect transmission errors.",

     "In CRC, a k-bit frame is regarded as the coefficient list of a polynomial with coefficients 0 and 1, ranging from x^(k−1) down to x^0.",

     "The generator polynomial G(x) is agreed upon by the sender and receiver and is used to generate and verify the CRC checksum.",

     "CRC checksum generation involves appending r zeros to the frame, dividing the resulting bit string by G(x) using modulo-2 division, and subtracting the remainder to form the transmitted checksummed frame.",

     "The MAC sublayer controls access to a shared communication medium and resolves multiple-access conflicts among stations.",

     "Static channel allocation assigns fixed frequencies or time slots to users using FDM or TDM, whereas dynamic channel allocation assigns resources according to user demand.",

     "The three major categories of multiple access protocols are Random Access Protocols, Controlled Access Protocols, and Channelization Protocols.",

     "In Pure ALOHA, stations transmit whenever they have data. If a collision occurs, the frame is lost and the station waits a random amount of time before retransmitting.",

     "Slotted ALOHA divides time into slots and allows transmission only at the beginning of a slot, reducing the probability of collisions compared to Pure ALOHA.",

     "CSMA operates on the carrier-sense principle, where a station listens to the channel before transmitting to determine whether it is idle or busy.",

     "Non-Persistent CSMA reduces collisions by waiting a random amount of time before sensing the channel again when it is busy, rather than continuously monitoring it.",

     "ARP (Address Resolution Protocol) maps a logical address such as an IP address to a physical address such as a MAC address.",

     "Logical addresses are used at the network layer for identifying hosts and routers, while physical addresses are local addresses used at the data-link or physical layer.",

     "ARP resolves a physical address by broadcasting an ARP request containing the target IP address and receiving a unicast ARP reply containing the target's physical address.",

     "An ARP request contains the sender's physical address, sender's IP address, target IP address, and a target physical address field filled with zeros.",

     "ARP caching stores recently resolved IP-to-MAC mappings so that future communications can use the stored mapping without generating new ARP broadcasts.",

     "The Ethernet preamble synchronizes the receiver with the incoming signal, while the Start of Frame Delimiter indicates that the actual frame is about to begin.",

     "In Binary Exponential Backoff, after each collision a station waits a randomly selected number of slot times from 0 to 2^i−1 before retransmitting, where i is the collision count. After 16 collisions, transmission failure is reported."
]

# --- 2. Collect the generated answers and context from your LangGraph pipeline ---
data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": ground_truths
}

print("Running 10 questions through the LangGraph pipeline...")
for q in test_questions:
    # Run our LangGraph app from the previous step
    # (Note: Assumes 'app' is defined and initialized earlier in your environment)
    result = app.invoke({"question": q})

    data["question"].append(q)
    data["answer"].append(result["generation"])
    data["contexts"].append(result["context"])


# --- 3. Convert to a HuggingFace Dataset (required by RAGAS) ---
dataset = Dataset.from_dict(data)

# --- 4. Initialize the evaluator models ---
print("Evaluating results with RAGAS (This may take a few minutes)...")
eval_llm = ChatGroq(model="llama-3.1-8b-instant")
eval_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# -> Wrap the Langchain objects so Ragas can process them without the kwargs error
# (Note: You may see a DeprecationWarning for these wrappers, but they are currently
# required to bridge Langchain and Ragas v0.2 smoothly).
ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)

# --- 5. Run the evaluation ---
evaluation_result = evaluate(
    dataset=dataset,
    metrics=[
         ContextRecall()  ,       # NEW: Instantiated with parentheses

   # NEW: Instantiated with parentheses
    ],
    llm=ragas_llm,               # Use wrapped LLM
    embeddings=ragas_embeddings  # Use wrapped Embeddings
)

# --- 6. Display the final scorecard ---
df = evaluation_result.to_pandas()
df.to_csv("context_recall_results.csv", index=False)

print(df)

# df.to_csv("rag_evaluation_results.csv", index=False) # Optional: save to file

/tmp/ipykernel_9629/3867670602.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_9629/3867670602.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_9629/3867670602.py:6: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, Answer

Running 10 questions through the LangGraph pipeline...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step running...
Retrieval step running...
Generation step ru

/tmp/ipykernel_9629/3867670602.py:115: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(eval_llm)
/tmp/ipykernel_9629/3867670602.py:116: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[9]: TimeoutError()


                                           user_input  \
0   What is the difference between error-correctin...   
1   What are the three linear systematic block cod...   
2            What is a Cyclic Redundancy Check (CRC)?   
3   How is a bit string represented in CRC polynom...   
4   What is the role of the generator polynomial G...   
5   What are the steps involved in CRC checksum ge...   
6   What is the purpose of the Medium Access Contr...   
7   What is the difference between static and dyna...   
8   What are the three major categories of multipl...   
9              How does Pure ALOHA handle collisions?   
10    How does Slotted ALOHA improve upon Pure ALOHA?   
11  What is the basic principle of Carrier Sense M...   
12  How does Non-Persistent CSMA reduce collisions...   
13                        What is the purpose of ARP?   
14  What is the difference between logical and phy...   
15  How does ARP resolve a physical address from a...   
16  What information is contain

In [ ]:
import pandas as pd

faithfulness = pd.read_csv("faithfulness_results.csv")
answer = pd.read_csv("answer_relevancy_results.csv")
context_precision = pd.read_csv("context_precision_results.csv")
context_recall = pd.read_csv("context_recall_results.csv")

# Keep only the metric columns from later runs
final_df = faithfulness.copy()

final_df["answer_relevancy"] = answer["answer_relevancy"]
final_df["context_precision"] = context_precision["context_precision"]
final_df["context_recall"] = context_recall["context_recall"]

display(final_df)

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the difference between error-correctin...,['•The other is to include only enough redunda...,The difference between error-correcting codes ...,Error-correcting codes include enough redundan...,1.000000,0.50,0.500000,0.50
1,What are the three linear systematic block cod...,['•We will examine three different error-detec...,The three linear systematic block codes discus...,The three linear systematic block codes discus...,1.000000,0.50,0.000000,1.00
2,What is a Cyclic Redundancy Check (CRC)?,"['Cyclic Redundancy Checks (CRCs).', '• The ch...",A Cyclic Redundancy Check (CRC) is a checksum ...,A Cyclic Redundancy Check (CRC) is an error-de...,1.000000,1.00,0.500000,1.00
3,How is a bit string represented in CRC polynom...,['•The polynomial code or CRC (Cyclic Redundan...,"In CRC polynomial coding, a bit string is repr...","In CRC, a k-bit frame is regarded as the coeff...",1.000000,0.75,1.000000,0.75
4,What is the role of the generator polynomial G...,['•When the polynomial code method is employed...,The generator polynomial G(x) is used to compu...,The generator polynomial G(x) is agreed upon b...,1.000000,1.00,1.000000,1.00
5,What are the steps involved in CRC checksum ge...,['•The algorithm for computing the checksum is...,The steps involved in CRC checksum generation ...,CRC checksum generation involves appending r z...,0.333333,1.00,1.000000,1.00
6,What is the purpose of the Medium Access Contr...,['THE MEDIUM ACCESS CONTROL SUBLAYER • A netwo...,The purpose of the Medium Access Control (MAC)...,The MAC sublayer controls access to a shared c...,0.500000,0.75,0.500000,1.00
7,What is the difference between static and dyna...,['• There are two different methods of channel...,The difference between static and dynamic chan...,Static channel allocation assigns fixed freque...,0.833333,1.00,1.000000,1.00
8,What are the three major categories of multipl...,['• Multiple Access Protocols • Many protocols...,The three major categories of multiple access ...,The three major categories of multiple access ...,1.000000,0.83,1.000000,1.00
9,How does Pure ALOHA handle collisions?,"['• In pure ALOHA, whenever any station transm...","In Pure ALOHA, when a collision occurs, the st...","In Pure ALOHA, stations transmit whenever they...",0.625000,1.00,1.000000,NaN


PHASE 4

In [ ]:
# DIAGNOSTIC — find what your vector store and embeddings objects are actually called

import builtins

print("=== Variables in builtins (saved across cells) ===")
builtin_vars = [name for name in dir(builtins) if not name.startswith("_")]
# Filter out the standard python builtins, show only the "extra" ones you've added
standard_builtins = set(dir(__builtins__)) if isinstance(__builtins__, dict) else set(dir(__builtins__))
custom = [v for v in builtin_vars if v not in standard_builtins]
print(custom if custom else "None found")

print("\n=== Variables in current notebook global scope ===")
import types
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    type_name = type(val).__name__
    if any(keyword in type_name.lower() or keyword in name.lower()
           for keyword in ["chroma", "embed", "collection", "client", "groq"]):
        print(f"  {name}  -->  type: {type_name}")

=== Variables in builtins (saved across cells) ===
None found

=== Variables in current notebook global scope ===
  GoogleGenerativeAIEmbeddings  -->  type: ModelMetaclass
  Chroma  -->  type: ABCMeta
  embedding_model  -->  type: GoogleGenerativeAIEmbeddings
  db  -->  type: Chroma
  ChatGroq  -->  type: ModelMetaclass
  GROQ_TOKEN  -->  type: str
  embedding_engine  -->  type: GoogleGenerativeAIEmbeddings
  chroma_db  -->  type: Chroma
  chat_model  -->  type: ChatGroq
  LangchainEmbeddingsWrapper  -->  type: DeprecationHelper
  judge_llm  -->  type: ChatGroq
  judge_embeddings  -->  type: GoogleGenerativeAIEmbeddings
  wrapped_embeddings  -->  type: LangchainEmbeddingsWrapper
  eval_llm  -->  type: ChatGroq
  eval_embeddings  -->  type: GoogleGenerativeAIEmbeddings
  ragas_embeddings  -->  type: LangchainEmbeddingsWrapper
  co  -->  type: Client


In [ ]:
# DIAGNOSTIC — inspect Phase 2 baseline CSVs before merging into comparison

import pandas as pd

csv_files = ["answer_relevancy.csv", "faithfulness.csv", "context_precision.csv", "context_recall.csv"]

for fname in csv_files:
    try:
        df = pd.read_csv(fname)
        print(f"=== {fname} ===")
        print(f"  shape: {df.shape}")
        print(f"  columns: {list(df.columns)}")
        print(df.head(3).to_string())
        print()
    except Exception as e:
        print(f"=== {fname} ===")
        print(f"  Error loading: {e}\n")

=== answer_relevancy.csv ===
  shape: (20, 5)
  columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'answer_relevancy']
                                                                         user_input                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [ ]:
# =============================================================================
# PHASE 4 (v3) — Hybrid Search (BM25 + Vector) + Cohere Re-ranking
# Fixes in this version:
#   - Real test questions (networking topics, matching your actual documents)
#   - Cohere rate-limit handling (trial key = 10 calls/min) with retry + backoff
#   - Comparison cell loads your 4 baseline CSVs instead of a missing eval_df
# =============================================================================

# CELL 10 — Install Phase 4 improvement libraries
!pip install rank-bm25 cohere -q
print("rank-bm25 and cohere installed")


# CELL 11 — Load Cohere API key from Colab Secrets

from google.colab import userdata
import os

os.environ["COHERE_API_KEY"] = userdata.get("COHERE_KEY")
print("Cohere API key loaded successfully")


# CELL 12 — Build a BM25 keyword index alongside the existing ChromaDB vector index

try:
    from rank_bm25 import BM25Okapi
    import re

    def tokenize(text: str) -> list:
        """Simple tokenizer: lowercase, strip punctuation, split on whitespace."""
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        return text.split()

    def _get_all_chunks(vectorstore):
        """Works whether the LangChain Chroma wrapper exposes .get() directly
        or only via the underlying ._collection."""
        try:
            return vectorstore.get(include=["documents", "metadatas"])
        except Exception:
            return vectorstore._collection.get(include=["documents", "metadatas"])

    print("Fetching all chunks from ChromaDB to build the BM25 index ...")
    all_data = _get_all_chunks(chroma_db)

    bm25_corpus = all_data["documents"]
    bm25_metadatas = all_data["metadatas"]
    bm25_ids = all_data["ids"]

    print(f"  Found {len(bm25_corpus)} chunks in chroma_db")

    tokenized_corpus = [tokenize(doc) for doc in bm25_corpus]
    bm25_index = BM25Okapi(tokenized_corpus)

    print(f"BM25 index built over {len(bm25_corpus)} chunks")

except Exception as e:
    print(f"Error building BM25 index: {e}")


# CELL 13 — Hybrid search (BM25 + vector) combined, then re-ranked by Cohere
# (with retry/backoff on 429 rate-limit errors — trial keys allow 10 calls/min)

try:
    import cohere
    import time as _time

    co = cohere.Client(os.environ["COHERE_API_KEY"])

    def _rerank_with_retry(query, documents, top_n, max_retries=5):
        """Calls co.rerank, retrying with backoff if we hit the trial rate limit."""
        for attempt in range(max_retries):
            try:
                return co.rerank(
                    model="rerank-english-v3.0",
                    query=query,
                    documents=documents,
                    top_n=top_n
                )
            except Exception as e:
                if "429" in str(e) or "rate" in str(e).lower():
                    wait = 8 * (attempt + 1)
                    print(f"   Cohere rate limit hit — waiting {wait}s before retry ({attempt+1}/{max_retries})")
                    _time.sleep(wait)
                else:
                    raise
        raise RuntimeError("Cohere rerank failed after max retries (rate limit)")

    def hybrid_retrieve(question: str, vector_k: int = 10, bm25_k: int = 10, final_k: int = 3) -> dict:
        # ── Step 1: Vector search (LangChain handles embedding internally) ─────
        vector_results = chroma_db.similarity_search(question, k=vector_k)
        vector_docs = [doc.page_content for doc in vector_results]
        vector_metas = [doc.metadata for doc in vector_results]

        # ── Step 2: BM25 keyword search ────────────────────────────────────────
        tokenized_query = tokenize(question)
        bm25_scores = bm25_index.get_scores(tokenized_query)
        top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:bm25_k]

        bm25_docs = [bm25_corpus[i] for i in top_bm25_idx]
        bm25_metas = [bm25_metadatas[i] for i in top_bm25_idx]

        # ── Step 3: Merge + de-duplicate (by chunk text) ────────────────────────
        seen = set()
        merged_docs = []
        merged_metas = []

        for doc, meta in zip(vector_docs + bm25_docs, vector_metas + bm25_metas):
            if doc not in seen:
                seen.add(doc)
                merged_docs.append(doc)
                merged_metas.append(meta)

        print(f"   Hybrid search: {len(vector_docs)} vector + {len(bm25_docs)} BM25 → {len(merged_docs)} unique chunks")

        # ── Step 4: Re-rank with Cohere (with retry on rate limit) ──────────────
        if len(merged_docs) == 0:
            return {"documents": [], "metadatas": []}

        rerank_response = _rerank_with_retry(
            query=question,
            documents=merged_docs,
            top_n=min(final_k, len(merged_docs))
        )

        # ── Step 5: Extract top final_k re-ranked chunks ────────────────────────
        final_docs = []
        final_metas = []
        for result in rerank_response.results:
            idx = result.index
            final_docs.append(merged_docs[idx])
            final_metas.append(merged_metas[idx])

        print(f"  Re-ranked down to top {len(final_docs)} chunks")

        return {"documents": final_docs, "metadatas": final_metas}

    print("hybrid_retrieve() function defined and ready")

except Exception as e:
    print(f"Error defining hybrid retrieval function: {e}")


# CELL 14 — Full improved RAG pipeline: hybrid search + re-ranking + chat_model generation

try:
    def generate_answer_v2(question: str, documents: list, metadatas: list) -> str:
        context = "\n\n---\n\n".join(
            [
                f"[Source: {m.get('source', 'Unknown')}]\n{doc}"
                for doc, m in zip(documents, metadatas)
            ]
        )

        prompt = f"""You are a helpful AI assistant. Answer the question using ONLY the context provided below.
Cite the source for every fact you state. If the context does not contain enough information, say so clearly.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

        response = chat_model.invoke(prompt)
        return response.content

    def ask_question_v2(question: str) -> dict:
        retrieval = hybrid_retrieve(question, vector_k=10, bm25_k=10, final_k=3)
        documents = retrieval["documents"]
        metadatas = retrieval["metadatas"]

        if not documents:
            return {
                "question": question,
                "answer": "No relevant chunks found.",
                "chunks": [],
                "sources": []
            }

        answer = generate_answer_v2(question, documents, metadatas)
        sources = [m.get("source", "Unknown") for m in metadatas]

        return {
            "question": question,
            "answer": answer,
            "chunks": documents,
            "sources": sources
        }

    print("Testing improved RAG pipeline (hybrid search + re-ranking) ...\n")
    test_result = ask_question_v2("What is a Cyclic Redundancy Check (CRC)?")

    print(f"\nQuestion : {test_result['question']}")
    print(f"\nAnswer  :\n{test_result['answer']}")
    print(f"\nSources  : {list(set(test_result['sources']))}")
    print(f"\nChunks   : {len(test_result['chunks'])} retrieved (after re-ranking)")

except Exception as e:
    print(f"Error testing improved pipeline: {e}")


# =============================================================================
# CELL 15 — Re-run the same 20 questions through ask_question_v2 (hybrid + re-ranked)
# and score each answer the same way Phase 2 did
# =============================================================================

try:
    import pandas as pd
    import time

    # ── Your actual 20 test questions (networking topics) ──────────────────────
    TEST_QUESTIONS = [
        "What is the difference between error-correcting codes and error-detecting codes?",
        "What are the three linear systematic block codes discussed for error detection?",
        "What is a Cyclic Redundancy Check (CRC)?",
        "How is a bit string represented in CRC polynomial coding?",
        "What is the role of the generator polynomial G(x) in CRC?",
        "What are the steps involved in CRC checksum generation?",
        "What is the purpose of the Medium Access Control (MAC) sublayer?",
        "What is the difference between static and dynamic channel allocation?",
        "What are the three major categories of multiple access protocols?",
        "How does Pure ALOHA handle collisions?",
        "How does Slotted ALOHA improve upon Pure ALOHA?",
        "What is the basic principle of Carrier Sense Multiple Access (CSMA)?",
        "How does Non-Persistent CSMA reduce collisions compared to 1-Persistent CSMA?",
        "What is the purpose of ARP?",
        "What is the difference between logical and physical addresses?",
        "How does ARP resolve a physical address from an IP address?",
        "What information is contained in an ARP request message?",
        "What is ARP caching and why is it useful?",
        "What is the purpose of the Ethernet preamble and Start of Frame Delimiter?",
        "How does the Binary Exponential Backoff algorithm work in Ethernet?",
    ]

    # ── Reuse the exact same scoring logic from Phase 2, via chat_model ────────
    def score_answer(question: str, answer: str, context: str) -> dict:
        scoring_prompt = f"""You are an expert RAG evaluator. Score the following RAG output strictly.

QUESTION: {question}

RETRIEVED CONTEXT:
{context[:1500]}

GENERATED ANSWER:
{answer[:800]}

Score each metric from 0.0 to 1.0 (two decimal places):
1. faithfulness       — Is the answer fully supported by the context? (1.0 = fully grounded, 0.0 = hallucinated)
2. answer_relevancy   — Does the answer directly address the question? (1.0 = perfectly relevant, 0.0 = off-topic)
3. context_precision  — Does the retrieved context contain information needed to answer? (1.0 = highly relevant context, 0.0 = irrelevant)

Reply in this EXACT format and nothing else:
faithfulness:
answer_relevancy:
context_precision: """

        for attempt in range(3):
            try:
                response = chat_model.invoke(scoring_prompt)
                raw = response.content.strip()

                scores = {}
                for line in raw.splitlines():
                    if "faithfulness:" in line:
                        scores["faithfulness"] = float(line.split(":")[1].strip())
                    elif "answer_relevancy:" in line:
                        scores["answer_relevancy"] = float(line.split(":")[1].strip())
                    elif "context_precision:" in line:
                        scores["context_precision"] = float(line.split(":")[1].strip())

                if len(scores) == 3:
                    return scores

            except Exception as e:
                if attempt < 2:
                    time.sleep(10)
                else:
                    print(f"Scoring failed after 3 attempts: {e}")

        return {"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0}

    # ── Run pipeline v2 + scoring for all 20 questions ──────────────────────────
    print("Running IMPROVED RAG pipeline (hybrid search + re-ranking) on 20 questions ...\n")

    records_v2 = []

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"  [{i:02d}/20] {question[:60]}{'...' if len(question) > 60 else ''}", end=" ")

        try:
            rag_result = ask_question_v2(question)
            answer = rag_result["answer"]
            chunks = rag_result["chunks"]
            sources = rag_result["sources"]
            context = "\n\n".join(chunks)

            scores = score_answer(question, answer, context)

            records_v2.append({
                "question": question,
                "answer": answer[:120] + "..." if len(answer) > 120 else answer,
                "sources": ", ".join(set(sources)),
                "faithfulness": scores["faithfulness"],
                "answer_relevancy": scores["answer_relevancy"],
                "context_precision": scores["context_precision"],
            })

            print(
                f"  F={scores['faithfulness']:.2f}  "
                f"AR={scores['answer_relevancy']:.2f}  "
                f"CP={scores['context_precision']:.2f}"
            )

        except Exception as e:
            print(f"Error: {e}")
            records_v2.append({
                "question": question,
                "answer": "Error",
                "sources": "",
                "faithfulness": 0.0,
                "answer_relevancy": 0.0,
                "context_precision": 0.0,
            })

        # Cohere trial key = 10 calls/min. Each question makes 1 rerank call,
        # so we space them out to stay safely under that limit.
        time.sleep(7)

    df_v2 = pd.DataFrame(records_v2)
    df_v2.index = df_v2.index + 1

    print("\n" + "=" * 90)
    print(" IMPROVED PIPELINE RESULTS — 20 TEST QUESTIONS (Hybrid Search + Re-ranking)")
    print("=" * 90)
    print(df_v2[["question", "faithfulness", "answer_relevancy", "context_precision"]].to_string())

    print("\n" + "=" * 90)
    print(" AGGREGATE SCORES (v2 — Improved)")
    print("=" * 90)
    print(f"  Faithfulness        : {df_v2['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy    : {df_v2['answer_relevancy'].mean():.3f}")
    print(f"  Context Precision   : {df_v2['context_precision'].mean():.3f}")
    print(f"  Overall Average     : {df_v2[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f}")

    eval_df_v2 = df_v2
    print("\nImproved results saved as eval_df_v2 — proceed to the before/after comparison cell")

except Exception as e:
    print(f"An unexpected error occurred: {e}")


# =============================================================================
# CELL 16 — Side-by-side before (Phase 2, from CSVs) vs after (Phase 4) comparison
# =============================================================================

try:
    import pandas as pd

    # ── Load Phase 2 baseline from the 4 Ragas CSVs ─────────────────────────────
    ar_df = pd.read_csv("answer_relevancy.csv")[["user_input", "answer_relevancy"]]
    f_df  = pd.read_csv("faithfulness.csv")[["user_input", "faithfulness"]]
    cp_df = pd.read_csv("context_precision.csv")[["user_input", "context_precision"]]

    # Merge the three metrics into one baseline table, keyed by question text
    df_before_full = ar_df.merge(f_df, on="user_input").merge(cp_df, on="user_input")

    comparison_rows = []

    for question in TEST_QUESTIONS:
        before_row = df_before_full[df_before_full["user_input"] == question]
        after_row  = eval_df_v2[eval_df_v2["question"] == question]

        if before_row.empty or after_row.empty:
            print(f"  Skipping (no match found): {question[:60]}")
            continue

        before = before_row.iloc[0]
        after  = after_row.iloc[0]

        comparison_rows.append({
            "question": question[:55] + "..." if len(question) > 55 else question,
            "faithfulness_before": before["faithfulness"],
            "faithfulness_after":  after["faithfulness"],
            "relevancy_before":    before["answer_relevancy"],
            "relevancy_after":     after["answer_relevancy"],
            "precision_before":    before["context_precision"],
            "precision_after":     after["context_precision"],
        })

    df_compare = pd.DataFrame(comparison_rows)
    df_compare.index = df_compare.index + 1

    print("=" * 110)
    print(" BEFORE (Phase 2) vs AFTER (Phase 4: Hybrid Search + Re-ranking) — 20 TEST QUESTIONS")
    print("=" * 110)
    print(df_compare.to_string())

    print("\n" + "=" * 110)
    print(" AGGREGATE COMPARISON")
    print("=" * 110)

    metrics = [
        ("Faithfulness",      "faithfulness_before",  "faithfulness_after"),
        ("Answer Relevancy",  "relevancy_before",      "relevancy_after"),
        ("Context Precision", "precision_before",      "precision_after"),
    ]

    for name, before_col, after_col in metrics:
        before_mean = df_compare[before_col].mean()
        after_mean  = df_compare[after_col].mean()
        delta = after_mean - before_mean
        arrow = "📈" if delta > 0 else ("📉" if delta < 0 else "➡️")
        print(f"  {name:<20} : {before_mean:.3f} → {after_mean:.3f}   ({arrow} {delta:+.3f})")

    overall_before = df_compare[["faithfulness_before", "relevancy_before", "precision_before"]].mean().mean()
    overall_after  = df_compare[["faithfulness_after",  "relevancy_after",  "precision_after"]].mean().mean()
    print(f"\n  {'Overall Average':<20} : {overall_before:.3f} → {overall_after:.3f}   ({overall_after - overall_before:+.3f})")

    deltas = {name: df_compare[after_col].mean() - df_compare[before_col].mean()
              for name, before_col, after_col in metrics}
    best_metric = max(deltas, key=deltas.get)
    print(f"\n  Most improved metric : {best_metric} (+{deltas[best_metric]:.3f})")

    still_failing = df_compare[
        (df_compare["faithfulness_after"] < 0.5) | (df_compare["precision_after"] < 0.5)
    ]

    print(f"\n    Questions still failing after improvement ({len(still_failing)}):")
    if len(still_failing) > 0:
        for _, row in still_failing.iterrows():
            print(f"     - {row['question']}")
    else:
        print("     None — all questions scored above 0.5 on both metrics")

    print("\nComparison table saved as df_compare")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

rank-bm25 and cohere installed
Cohere API key loaded successfully
Fetching all chunks from ChromaDB to build the BM25 index ...
  Found 191 chunks in chroma_db
BM25 index built over 191 chunks
hybrid_retrieve() function defined and ready
Testing improved RAG pipeline (hybrid search + re-ranking) ...

   Hybrid search: 10 vector + 10 BM25 → 15 unique chunks
  Re-ranked down to top 3 chunks

Question : What is a Cyclic Redundancy Check (CRC)?

Answer  :
A Cyclic Redundancy Check (CRC) is a polynomial code that is in widespread use, where bit strings are treated as representations of polynomials with coefficients of 0 and 1 only [Source: CRC.pdf]. It is also an algorithm used for error detection [Source: ETHERNET.pdf].

Sources  : ['CRC.pdf', 'ETHERNET.pdf']

Chunks   : 3 retrieved (after re-ranking)
Running IMPROVED RAG pipeline (hybrid search + re-ranking) on 20 questions ...

  [01/20] What is the difference between error-correcting codes and er...    Hybrid search: 10 vector + 10 BM25

In [ ]:
df_compare.to_csv("phase4_comparison.csv", index=False)
